<a href="https://colab.research.google.com/github/rathans48/flyrank-ml-internship-starter/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

One row = one content item, aggregated over one calendar month (a "content-month" summary). The underlying warehouse table (fact_content_daily_performance) is daily-grain, but for Lane 3 (archetype clustering) I roll each page's daily rows up to a single monthly snapshot — one row per page, per month.

Time window: March 2026 (month=2026-03), a mid-panel month. Deliberately not the _sample table, which is June 2026 — the panel's final month, reserved as a sealed test month per the data skill.

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

%pip -q install duckdb
import duckdb
from google.colab import userdata

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{userdata.get('HF_TOKEN')}')")

DAILY = "read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')"

con.sql(f"""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) c
    FROM {DAILY}
    GROUP BY report_date, client_hash_id, content_hash_id
    HAVING c > 1
    LIMIT 5
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,report_date,client_hash_id,content_hash_id,c


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

Feature (knowable by end of March, safe to use for clustering):

gsc_impressions, gsc_clicks, gsc_avg_position — daily search measurements, fully observed by month-end
ga4_engaged_sessions — only where ga4_data_available IS TRUE
content_age_days — derived from creation date, always in the past

Label / proxy: None. Clustering has no target. The only label-shaped field I touch at all is trend_pct, and only to deliberately demonstrate why it must never be a feature (section 4).

Context (grouping/joining only, never fed to a model): content_hash_id, client_hash_id, keyword_hash_id/url_hash_id if joined in later.

Excluded:

- trend_direction, trend_pct — NOT present in this warehouse table at all (DESCRIBE confirmed the schema above has no such columns; these fields only exist in the small starter CSV, not fact_content_daily_performance). Named here for completeness — if they were present, they'd be label-derived and excluded for the same reason the starter-CSV skill flags them: trend_direction/trend_pct define what a decline label would be computed from, so using them as clustering input would be circular.
- Any product-decision fields (health_score, priority_score, action_type) — confirmed absent from the schema (see DESCRIBE output above), so nothing to exclude by hand, but named here for completeness.
- Raw query/URL/title text — never present in this release; excluded because de-anonymizing content is out of scope even if it existed.

In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

CONTENT = "read_parquet('hf://datasets/FlyRank/internship-warehouse/dim_content.parquet')"

# Confirm the excluded product-decision fields genuinely aren't present as columns
con.sql(f"DESCRIBE SELECT * FROM {DAILY} LIMIT 1").df()

,column_name,column_type,null,key,default,extra
0,report_date,DATE,YES,None,None,None
1,client_hash_id,VARCHAR,YES,None,None,None
2,content_hash_id,VARCHAR,YES,None,None,None
3,client_has_gsc,BOOLEAN,YES,None,None,None
4,client_has_ga4,BOOLEAN,YES,None,None,None
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None
7,gsc_impressions,BIGINT,YES,None,None,None
8,gsc_clicks,BIGINT,YES,None,None,None
9,gsc_sum_position,BIGINT,YES,None,None,None


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

Four claims, four queries — all verified:

1. Grain: the grain-probe (section 1) returned zero duplicate rows for report_date x client_hash_id x content_hash_id — confirms one row per content item per client per day.
2. Counts + window: this March 2026 slice contains 9,841,378 rows, spanning exactly 2026-03-01 to 2026-03-31, across 331,437 distinct content items — roughly 64% of the warehouse's 519,606 total content items showed any activity this month.
3. Missingness: ga4_engaged_sessions is null in 30.7% of rows overall, but 0% null when ga4_data_available IS TRUE — confirming the gaps follow the tracking-availability flag rather than being random.
4. Window confirmation: MIN/MAX dates above fall entirely inside March 2026, with no spillover into February or April.


In [13]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Counts + window
con.sql(f"""
    SELECT COUNT(*) AS n_rows, MIN(report_date) AS min_d, MAX(report_date) AS max_d,
           COUNT(DISTINCT content_hash_id) AS n_content
    FROM {DAILY}
""").df()

con.sql(f"""
    SELECT
        AVG(CASE WHEN ga4_engaged_sessions IS NULL THEN 1.0 ELSE 0 END) AS pct_null_overall,
        AVG(CASE WHEN ga4_data_available IS TRUE AND ga4_engaged_sessions IS NULL
                 THEN 1.0 ELSE 0 END) AS pct_null_when_available
    FROM {DAILY}
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,pct_null_overall,pct_null_when_available
0,0.30674,0.0


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

This data can never tell me:

Causation. Even if clusters separate cleanly, I can't say why a page ended up in a group, or that any action (rewrite, refresh) would change its cluster — that needs an experiment, not this data.
A complete panel. History depth differs wildly per client (dim_clients.gsc_data_start/ga4_data_start) — March 2026 rows are missing entirely for clients whose tracking started later, not zero-valued. Any archetype found here is proven only for clients with sufficient March history.
True engagement for GSC-only rows. Rows before a client's ga4_data_start carry search data only; treating their zero-filled GA4 columns as real zeros would be a mistake — that's why ga4_data_available IS TRUE gates every engagement feature above.
AI-referral or algorithm insight. Not part of this slice at all this week, and out of scope for Lane 3 regardless.

Concretely: of the 55 clients active in March 2026, only 26 (47%) had GA4 tracking running for the entire month — the remainder have partial or absent engagement data this month, which is exactly why every engagement feature above is gated on ga4_data_available IS TRUE rather than treated as a raw zero.

In [14]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

CLIENTS = "read_parquet('hf://datasets/FlyRank/internship-warehouse/dim_clients.parquet')"

# How many of the clients present in March data actually had ga4 tracking active for all of March?
con.sql(f"""
    SELECT
        COUNT(DISTINCT d.client_hash_id) AS clients_in_march,
        COUNT(DISTINCT CASE WHEN c.ga4_data_start <= DATE '2026-03-01'
                            THEN d.client_hash_id END) AS clients_with_full_ga4_march
    FROM {DAILY} d
    JOIN {CLIENTS} c ON d.client_hash_id = c.client_hash_id
""").df()

,clients_in_march,clients_with_full_ga4_march
0,55,26


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.